In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# 1. Cargar clave API de forma segura
load_dotenv()

# 2. Cargar las actas y reglamentos de la carpeta
print("Cargando documentos...")
loader = PyPDFDirectoryLoader("data/actas")
documentos = loader.load()

# 3. Dividir los textos en fragmentos procesables
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
fragmentos = text_splitter.split_documents(documentos)

# 4. Crear los embeddings y guardar en la base de datos local (Chroma)
print("Creando base de datos vectorial...")
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=fragmentos, 
    embedding=embedding_function, 
    persist_directory="./chroma_db" # Se guarda localmente en esta carpeta
)

# 5. Configurar el retriever con un umbral de similitud estricto
# k=10 trae 10 fragmentos más relevantes para cada consulta, lo que permite un contexto más amplio para la respuesta.
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
print("¡Base de datos lista!")

C:\Users\Gabriel\AppData\Local\Temp\ipykernel_15312\1846096485.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Cargando documentos...
Creando base de datos vectorial...


C:\Users\Gabriel\AppData\Local\Temp\ipykernel_15312\1846096485.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

¡Base de datos lista!


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# 1. Cargar el entorno local
load_dotenv()
mi_clave_api = os.getenv("GOOGLE_API_KEY")

# 2. Instanciar Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", 
    temperature=0, 
    api_key=mi_clave_api
)

# 3. Actualizar retriever a k=10 para mayor alcance
retriever_actualizado = vectorstore.as_retriever(search_kwargs={"k": 10})

# 4. Prompt del sistema
system_prompt = (
    "Eres el asistente virtual oficial de la cooperativa de vivienda Los Cerezos. "
    "Tu rol es atender a los vecinos con un tono amable, empático, claro y servicial.\n\n"
    "Reglas estrictas de respuesta:\n"
    "1. Si la respuesta está en el contexto, explícala de forma natural, estructurada y conversacional.\n"
    "2. Si la información solicitada NO se encuentra en los documentos oficiales, "
    "NUNCA inventes datos, pero **evita sonar robótico**. En su lugar, responde con esta estructura cálida:\n"
    "'Lamento informarle que ese detalle específico no figura en los registros ni actas actuales de nuestra cooperativa. "
    "Para entregarle una orientación adecuada, le sugiero acercarse directamente a la mesa de consultas local de la directiva o comunicarse con la administración.'\n\n"
    "Contexto oficial disponible:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 5. Ensamblar la cadena
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever_actualizado, question_answer_chain)

# 6. Lista de prueba
preguntas_prueba = [
    "¿Cuál es el monto de las multas aplicadas por infracciones a la convivencia y en qué artículo se basan?",
    "¿Qué empresa está a cargo de la administración del condominio y quién la representa?",
    "¿Se mencionó algún plan o medida respecto al sector de los estacionamientos de visita?",
    "¿Qué soluciones se propusieron para enfrentar los problemas de caja o morosidad?",
    "¿Cuál es el horario de descanso establecido para los días viernes y vísperas de festivo?"
]

print("--- INICIANDO AGENTE DE LA COOPERATIVA ---\n")

for pregunta in preguntas_prueba:
    print(f"Vecino: {pregunta}")
    respuesta = rag_chain.invoke({"input": pregunta})
    print(f"Agente: {respuesta['answer']}\n")
    print("-" * 40 + "\n")






--- INICIANDO AGENTE DE LA COOPERATIVA ---

Vecino: ¿Cuál es el monto de las multas aplicadas por infracciones a la convivencia y en qué artículo se basan?


c:\Users\Gabriel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Agente: ¡Hola! Con mucho gusto le comparto la información sobre las multas por infracciones a la convivencia en nuestra cooperativa. Según los registros oficiales, los montos y sanciones se estructuran de la siguiente manera:

* **Primera infracción:** Amonestación por escrito, sin multa pecuniaria.
* **Segunda infracción:** Multa de 0,3 UF.
* **Tercera infracción:** Multa de 0,6 UF.
* **Cuarta infracción y siguientes:** Multa de 1,0 UF, sin perjuicio de que el Comité de Administración pueda ejercer las acciones que correspondan ante el Juzgado de Policía Local de Cerro Navia.

Además, en las actas de la Asamblea se contemplan casos específicos relacionados con distintos artículos del reglamento, por ejemplo:
* **Ruidos molestos** (Artículo 24°, letra b): Se ha aplicado una multa de 0,3 UF en casos específicos de segunda infracción.
* **Mascotas** (Artículo 27°): Se ha aplicado una multa de 0,3 UF por infracciones relacionadas con el cuidado y manejo de mascotas en zonas comunes.
* **E

c:\Users\Gabriel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Agente: ¡Hola! Con mucho gusto le comparto esa información. 

La administración del condominio está a cargo de la empresa **“Espinoza y Cía. Administración de Condominios Limitada”**, y es representada por el Administrador del condominio, don **Cristián Espinoza Ramírez**.

¿Hay alguna otra consulta en la que pueda ayudarle el día de hoy?

----------------------------------------

Vecino: ¿Se mencionó algún plan o medida respecto al sector de los estacionamientos de visita?


...

Agente: ¡Hola! Con mucho gusto le comparto la información sobre los estacionamientos de visita que figura en nuestros registros:

Durante el desarrollo de las reuniones, se informó que el Departamento 611 incurrió nuevamente en el uso no registrado de un estacionamiento de visita, acumulando así una segunda infracción en el período. 

Debido a esto, el Comité advirtió que, en caso de reiterarse esta conducta, se procederá al retiro del vehículo mediante una grúa y a costa del infractor, una medida que se encuentra autorizada por el artículo 34° de nuestro Reglamento.

¿Hay algún otro tema sobre la convivencia en el condominio en el que le pueda ayudar?

----------------------------------------

Vecino: ¿Qué soluciones se propusieron para enfrentar los problemas de caja o morosidad?


c:\Users\Gabriel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Agente: Lamento informarle que ese detalle específico no figura en los registros ni actas actuales de nuestra cooperativa. Para entregarle una orientación adecuada, le sugiero acercarse directamente a la mesa de consultas local de la directiva o comunicarse con la administración.

----------------------------------------

Vecino: ¿Cuál es el horario de descanso establecido para los días viernes y vísperas de festivo?


c:\Users\Gabriel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Agente: ¡Hola! Con mucho gusto le comparto la información sobre los horarios de descanso en nuestra cooperativa.

Según el Artículo 22° de nuestro reglamento, para los días **viernes, sábado y vísperas de festivo**, el horario de descanso establecido comprende desde las **23:30 hasta las 09:00 horas** del día siguiente. 

Durante este periodo, le recordamos que está estrictamente prohibido producir o permitir ruidos, música, uso de herramientas, celebraciones u otras actividades que puedan perturbar el descanso de nuestros vecinos. 

¿Hay algo más en lo que pueda ayudarle hoy?

----------------------------------------

